# Lane Theory 03: Individual Diagnosis
Loads trained artifacts from Notebook 02, applies the same feature
engineering pipeline to individual swimmer data, and runs the
cluster-matching + optimization diagnosis

In [3]:
import pandas as pd
import numpy as np
import joblib
import pickle
import sys

sys.path.append("../src")
from feature_engineering import engineer_features
from optimization import (
    primitive_cols, feature_cols,
    get_cluster_bounds, get_nn_bounds, get_top_levers, optimize_swimmer,
    diagnose_swimmer, print_report,
)

pd.set_option("display.max_columns", None)

models = joblib.load("../models/cluster_models.joblib")
with open("../models/shap_values_dict.pkl", "rb") as f:
    shap_values_dict = pickle.load(f)

elite_features = pd.read_csv("../data/elite/processed/100m_freestyle_features.csv")

print(f"Loaded {len(models)} models, SHAP for {len(shap_values_dict)} clusters, "
      f"{len(elite_features)} elite rows")

Loaded 4 models, SHAP for 4 clusters, 40 elite rows


In [4]:
import optimization

print(dir(optimization))

['__builtins__', '__cached__', '__doc__', '__file__', '__loader__', '__name__', '__package__', '__spec__', 'derived_cols', 'diagnose_swimmer', 'differential_evolution', 'feature_cols', 'get_cluster_bounds', 'get_nn_bounds', 'get_top_levers', 'np', 'optimize_swimmer', 'pd', 'primitive_cols', 'print_report']


## Auto-ID generation (for future users)

In [5]:
INDIVIDUALS_RAW_PATH = "../data/individuals/raw/individuals_raw.csv"

def get_next_athlete_id(existing_df=None):
    if existing_df is None or existing_df.empty:
        return "U001"
    existing_nums = existing_df["athlete_id"].str.extract(r"U(\d+)")[0].astype(int)
    next_num = existing_nums.max() + 1
    return f"U{next_num:03d}"

def get_next_race_id(athlete_id, existing_df=None):
    if existing_df is None or existing_df.empty:
        return f"{athlete_id}-R01"
    athlete_races = existing_df[existing_df["athlete_id"] == athlete_id]
    if athlete_races.empty:
        return f"{athlete_id}-R01"
    existing_nums = athlete_races["race_id"].str.extract(r"-R(\d+)")[0].astype(int)
    next_num = existing_nums.max() + 1
    return f"{athlete_id}-R{next_num:02d}"

## Load individual data, apply feature engineering

In [6]:
individuals_raw = pd.read_csv("../data/individuals/raw/individuals_raw.csv")
individuals_features = engineer_features(individuals_raw)

individuals_features.to_csv("../data/individuals/processed/individuals_features.csv", index=False)
individuals_features

,athlete_id,race_id,name,gender,height_cm,date_of_birth,race_date,final_time_sec,pb_50m_seconds,l1_reaction_time,l1_breakout_distance,l1_breakout_time,l1_stroke_count,l1_total_time,l2_breakout_distance,l2_breakout_time,l2_stroke_count,l2_total_time,l1_split_25m,l2_split_25m,age_at_race,l1_underwater_speed,l1_surface_distance,l1_surface_time,l1_surface_speed,l1_stroke_length,l1_stroke_rate,l1_swolf,l2_underwater_speed,l2_surface_distance,l2_surface_time,l2_surface_speed,l2_stroke_length,l2_stroke_rate,l2_swolf,l1_first25_speed,l1_second25_speed,intra_lap1_fade,l2_first25_speed,l2_second25_speed,intra_lap2_fade,finish_vs_fresh_ratio,pacing_delta,front_end_pct,stroke_length_drop,stroke_rate_change,swolf_change,breakout_drop
0,U001,U001-R01,Daniel Siahaan,male,168,2004-07-27,2021-09-02,57.53,26.5,0.67,12,4.6,37,27.81,6.5,2.7,45,29.72,12.3,14.2,17.100616,2.608696,38.0,23.21,1.637225,1.027027,95.648427,60.21,2.407407,43.5,27.02,1.609919,0.966667,99.925981,72.02,2.03252,1.611863,0.420657,1.760563,1.610825,0.149739,0.792526,1.91,1.049434,0.06036,4.277553,11.81,5.5


## Select a user's race and assign to nearest elite height cluster

In [7]:
def assign_height_cluster(height, gender, elite_df):
    """Assign an individual to the elite height cluster (per gender) whose
    mean height is closest to theirs."""
    gender_clusters = elite_df[elite_df["gender"] == gender].groupby("height_group")["height_cm"].mean()
    closest_group = (gender_clusters - height).abs().idxmin()
    return closest_group, gender_clusters


def prepare_user_race(race_id, individuals_df, elite_df):
    """Pull one race by race_id, assign its height cluster, return the row."""
    row = individuals_df[individuals_df["race_id"] == race_id].iloc[0].copy()
    assigned_group, cluster_means = assign_height_cluster(row["height_cm"], row["gender"], elite_df)
    row["height_group"] = assigned_group
    return row, cluster_means


# Select which race to diagnose (swap this race_id for any user/race later)
selected_race_id = "U001-R01"
user_row, cluster_means = prepare_user_race(selected_race_id, individuals_features, elite_features)

print(f"{user_row['name']}: height {user_row['height_cm']}cm, gender {user_row['gender']}")
print(f"Cluster means: {cluster_means.to_dict()}")
print(f"Assigned to: {user_row['height_group']}")

Daniel Siahaan: height 168cm, gender male
Cluster means: {'shorter': 188.63636363636363, 'taller': 196.44444444444446}
Assigned to: shorter


**Observations — important limitation:**

Dino (168cm) is assigned to the male "shorter" cluster by nearest-mean
logic, but that cluster's actual observed range is 185–191cm — Dino sits
17cm below the shortest elite swimmer in the entire dataset (both
clusters combined). This is a genuine extrapolation problem, not a minor
edge case: the benchmark model and optimization bounds were learned
entirely from swimmers with a fundamentally different height range, and
biomechanics don't scale linearly with height (e.g., a 168cm swimmer's
natural stroke length ceiling is very unlikely to reach the 185-191cm
cluster's observed 1.14-1.41m range).

**This is a known and honest limitation of the current dataset, not a bug
to fix in code** — the elite dataset simply doesn't include swimmers in
Dino's height range. The diagnosis that follows should be interpreted
cautiously and explicitly flagged in the output/README as extrapolated
beyond the training data's support, rather than presented as a confident
recommendation.

## Run diagnosis on individual

In [8]:
def check_extrapolation(height, gender, height_group, elite_df):
    mask = (elite_df["gender"] == gender) & (elite_df["height_group"] == height_group)
    cluster_min = elite_df.loc[mask, "height_cm"].min()
    cluster_max = elite_df.loc[mask, "height_cm"].max()
    if height < cluster_min or height > cluster_max:
        gap = min(abs(height - cluster_min), abs(height - cluster_max))
        return True, cluster_min, cluster_max, gap
    return False, cluster_min, cluster_max, 0


def run_diagnosis(user_row, elite_df, models, shap_values_dict, primitive_cols, feature_cols):
    is_extrapolated, c_min, c_max, gap = check_extrapolation(
        user_row["height_cm"], user_row["gender"], user_row["height_group"], elite_df
    )

    if is_extrapolated:
        print("WARNING: Extrapolation beyond training data")
        print(f"{user_row['name']}'s height ({user_row['height_cm']}cm) falls outside the "
              f"assigned cluster's observed range ({c_min}-{c_max}cm) by {gap:.0f}cm.")
        print("This diagnosis should be treated as exploratory, not a confident recommendation.\n")

    report = diagnose_swimmer(user_row, elite_df, models, shap_values_dict, primitive_cols, feature_cols)
    return report


user_report = run_diagnosis(user_row, elite_features, models, shap_values_dict, primitive_cols, feature_cols)
print_report(user_report)

Daniel Siahaan's height (168cm) falls outside the assigned cluster's observed range (185-191cm) by 17cm.
This diagnosis should be treated as exploratory, not a confident recommendation.

Daniel Siahaan (male, shorter)
Benchmarked against nearest 5 by height: Cameron Mcevoy, Guilherme Caribe, Caeleb Dressel, Flynn Southam, Fredrick Bousquet
Predicted rank: 6.27 -> 4.40

  l1_underwater_speed: 2.609 -> 3.349  (neighbor range: 3.000-3.568)
  l1_stroke_rate: 95.648 -> 98.416  (neighbor range: 92.751-106.223)
  l1_stroke_length: 1.027 -> 1.233  (neighbor range: 1.150-1.262)
  l2_underwater_speed: 2.407 -> 2.633  (neighbor range: 2.297-2.786)
  l1_reaction_time: 0.670 -> 0.629  (neighbor range: 0.610-0.710)


**Observations:**

Switching to nearest-neighbor bounds (5 closest elites by height, rather
than the full 11-swimmer cluster) produced meaningfully tighter, more
locally-grounded ranges — e.g., `l1_stroke_length` neighbor range
(1.150–1.262) is narrower than the full cluster range (1.150–1.333), and
`l1_underwater_speed` similarly tightens (3.000–3.568 vs. 3.000–3.935).
The optimized targets and predicted-rank improvement (6.27 → 4.40) stayed
nearly identical to the full-cluster run — reassuring, since it shows the
earlier result wasn't an artifact of an unrepresentative wide range, and
the 5 nearest neighbors (Mcevoy, Caribe, Dressel, Southam, Bousquet) are a
much more defensible comparison group for a 168cm swimmer than the full
185–191cm cluster.

**One nuance worth noting:** `predicted_rank` itself (6.27) still comes
from the Random Forest trained on the full male-shorter cluster — height
isn't actually a direct model input (only stroke/pacing features are), so
this prediction reflects the model's general relationship between stroke
mechanics and rank *within that cluster's training distribution*, not
something the nearest-neighbor bounds change. The height-based
extrapolation warning still applies to the underlying model call, even
though the optimization *bounds* are now well-grounded.

**Conclusion for Phase 7:** nearest-neighbor bounds are the right
approach and should be the default going forward — genuinely more
defensible than cluster-wide bounds for anyone far from their cluster's
center, and functionally equivalent for anyone near it.